
Is invariant inference what I'm really missing?

https://github.com/gipsyh/rIC3 can I install this
cargo +nightly install rIC3

https://github.com/agurfinkel/spacer-on-jupyter/blob/master/Dagstuhl2019.ipynb


In [ ]:
def solve_horn(chc, pp=False, q3=False, max_unfold=10, verbosity=0):
    z3.set_param(verbose=verbosity)

    s = z3.SolverFor('HORN')
    s.set('engine', 'spacer')
    s.set('spacer.order_children', 2)
    if not pp:
        s.set('xform.inline_eager', False)
        s.set('xform.inline_linear', False)
        s.set('xform.slice', False)

    if max_unfold > 0:
        s.set('spacer.max_level', max_unfold)

    if q3:
        # allow quantified variables in pobs
        s.set('spacer.ground_pobs', False)
        # enable quantified generalization
        s.set('spacer.q3.use_qgen', True)

    # add constraints to solver
    s.add(chc)
    if verbosity > 0:
        print(s.sexpr())
    # run solver
    res = s.check()
    # extract model or proof
    answer = None
    if res == z3.sat:
        answer = s.model()
    elif res == z3.unsat:
        answer = s.proof()
    return res, answer



In [13]:

from kdrag.all import *
def HornSolver():
    s = smt.SolverFor('HORN')
    s.set('engine', 'spacer')
    s.set('spacer.order_children', 2)
    return s

HornSolver()


def interpolate(vs, shared,A,B):
    f = smt.FreshFunction(*[v.sort() for v in shared], smt.BoolSort())
    C = f(*shared)
    s = HornSolver()
    s.add(smt.ForAll(vs, A, C))
    s.add(smt.ForAll(vs, C, smt.Not(B)))
    res = s.check()
    if res == smt.sat:
        return s.model().eval(C)
    else:
        raise Exception("No interpolant found", res)

a, b, x, y = smt.Ints('a b x y')
itp = interpolate([a,b,x,y], [b,a], smt.And(a < x, x < b), z3.And(b < a))
itp

Exists(k!0, And(Not(k!0 <= a), Not(b <= k!0)))

In [14]:
def invariant(st, stnext, init, trans, assertion):
    s = HornSolver()
    inv = smt.FreshFunction(*[v.sort() for v in s] , smt.BoolSort())
    s.add(smt.ForAll(st, init, inv(*st)))
    s.add(smt.ForAll(st + stnext, smt.And(inv(*st), trans), inv(*stnext)))
    s.add(smt.ForAll(st, inv(*st), assertion))
    res = s.check()
    if res == smt.sat:
        return s.model().eval(inv)
    else:
        raise Exception("No invariant found", res)

def invariant2(st, init, step, assertion):
    s = HornSolver()
    inv = smt.FreshFunction(*[v.sort() for v in s] , smt.BoolSort())
    s.add(smt.ForAll(st, init, inv(*st)))
    s.add(smt.ForAll(st, inv(*st)), inv(step(*st)))
    s.add(smt.ForAll(st, inv(*st), assertion))
    res = s.check()
    if res == smt.sat:
        return s.model().eval(inv)
    else:
        raise Exception("No invariant found", res)



In [ ]:
from z3 import *


s = SolverFor("HORN")





In [ ]:
from kdrag.all import *






Inductive theorem proving
Alternative induction principles. How to derive. How to think about?
Could I use spacer to get them?

n, n+1 recursion

ACL2
bundy
rippling


